### 08. Climate Dataset Harmonization: NUTS-2 to NUTS-3 (Kreise)

Spatial Harmonization of Climate Indicators for District-Level Climate & ESG Risk Modelling

#### Data Inputs

This notebook uses the following input datasets:

- Harmonized NUTS-2 climate dataset
- Germany NUTS-2 administrative boundaries
- Germany NUTS-3 (Kreise) administrative boundaries

**Objective**

This notebook harmonizes climate indicators from the NUTS-2 regional level to the finer NUTS-3 (Kreise) district level in Germany using area-weighted spatial interpolation.

**Why do this**

The climate risk modeling framework requires climate indicators at the district (Kreise / NUTS-3) level.

Because the climate indicators are available only at the NUTS-2 regional level, they must be spatially disaggregated to NUTS-3 before they can be integrated with district-level socio-economic indicators for climate and ESG risk modelling.

**Approach**

1. Load and validate the NUTS-2 climate dataset.
2. Load the NUTS-2 and NUTS-3 administrative boundaries.
3. Ensure both datasets share a common coordinate reference system (CRS).
4. Compute spatial intersections and area-weighting factors.
5. Disaggregate climate indicators from NUTS-2 to NUTS-3 using area-weighted interpolation.
6. Validate spatial coverage and aggregation consistency.
7. Export the harmonized NUTS-3 climate dataset.

**Outputs**

- Harmonized climate dataset at the NUTS-3 (Kreise) level (CSV/Parquet)
- Spatial validation and quality assessment diagnostics
- Area-weighting lookup table (optional, for reproducibility)

# Part I — Build and Validate the Spatial Lookup Table

### Import Libraries

In [8]:
# ============================================================
# 0. Import libraries and settings
# ============================================================

import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from pathlib import Path
from shapely.geometry import Polygon

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Display settings
# ------------------------------------------------------------

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

BASE_DIR = Path.cwd().parent          # Project root
RAW_CLIMATE_DIR = BASE_DIR / "data" / "raw" / "climate"

print("Project directory :", BASE_DIR)
print("Climate data path :", RAW_CLIMATE_DIR)
print("Path exists       :", RAW_CLIMATE_DIR.exists())

Project directory : d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany
Climate data path : d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\raw\climate
Path exists       : True


##### Create Germany-only NUTS Boundary Files

In [11]:
# ============================================================
# 1. Create Germany-only NUTS boundary files
# ============================================================

# Input Europe-wide NUTS boundary files
nuts2_europe_path = RAW_CLIMATE_DIR / "germany" / "NUTS_RG_01M_2021_4326_LEVL_2.shp"
nuts3_europe_path = RAW_CLIMATE_DIR / "germany" / "NUTS_RG_01M_2021_4326_LEVL_3.shp"

# Output Germany-only NUTS boundary files
nuts2_de_path = RAW_CLIMATE_DIR / "germany" / "nuts2_germany_2021.shp"
nuts3_de_path = RAW_CLIMATE_DIR / "germany" / "nuts3_germany_2021.shp"

# Check that input files exist
print("NUTS-2 Europe file exists:", nuts2_europe_path.exists())
print("NUTS-3 Europe file exists:", nuts3_europe_path.exists())

# Load Europe-wide NUTS files
nuts2 = gpd.read_file(nuts2_europe_path)
nuts3 = gpd.read_file(nuts3_europe_path)

# Filter Germany only
nuts2_de = nuts2[nuts2["CNTR_CODE"] == "DE"].copy()
nuts3_de = nuts3[nuts3["CNTR_CODE"] == "DE"].copy()

# Print diagnostics
print("\nEurope NUTS-2 shape:", nuts2.shape)
print("Germany NUTS-2 shape:", nuts2_de.shape)

print("\nEurope NUTS-3 shape:", nuts3.shape)
print("Germany NUTS-3 shape:", nuts3_de.shape)

print("\nNUTS-2 CRS:", nuts2_de.crs)
print("NUTS-3 CRS:", nuts3_de.crs)

# Save Germany-only boundary files
nuts2_de.to_file(nuts2_de_path)
nuts3_de.to_file(nuts3_de_path)

print("\nSaved Germany-only NUTS-2 to:", nuts2_de_path)
print("Saved Germany-only NUTS-3 to:", nuts3_de_path)

NUTS-2 Europe file exists: True
NUTS-3 Europe file exists: True

Europe NUTS-2 shape: (334, 18)
Germany NUTS-2 shape: (38, 18)

Europe NUTS-3 shape: (1514, 18)
Germany NUTS-3 shape: (401, 18)

NUTS-2 CRS: EPSG:4326
NUTS-3 CRS: EPSG:4326

Saved Germany-only NUTS-2 to: d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\raw\climate\germany\nuts2_germany_2021.shp
Saved Germany-only NUTS-3 to: d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\raw\climate\germany\nuts3_germany_2021.shp


##### Load Germany NUTS Administrative Boundaries

In [13]:
# ============================================================
# 2. Load Germany NUTS administrative boundaries
# ============================================================

nuts2_path = RAW_CLIMATE_DIR / "germany" / "nuts2_germany_2021.shp"
nuts3_path = RAW_CLIMATE_DIR / "germany" / "nuts3_germany_2021.shp"

nuts2 = gpd.read_file(nuts2_path)
nuts3 = gpd.read_file(nuts3_path)

print(f"NUTS-2 regions: {len(nuts2)}")
print(f"NUTS-3 districts: {len(nuts3)}")

display(nuts2.head())
display(nuts3.head())

NUTS-2 regions: 38
NUTS-3 districts: 401


,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,NAME_ENGL,NAME_FREN,ISO3_CODE,SVRG_UN,CAPT,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry
0,DE12,2,DE,Karlsruhe,Karlsruhe,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((9.45238 49.64864, 9.47674 49.6..."
1,DE14,2,DE,Tübingen,Tübingen,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((9.96528 48.59584, 9.9727 48.59462, 9..."
2,DE13,2,DE,Freiburg,Freiburg,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((8.00179 48.68692, 8.01441 48.6..."
3,DE21,2,DE,Oberbayern,Oberbayern,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((11.42642 49.06797, 11.44318 49..."
4,DEF0,2,DE,Schleswig-Holstein,Schleswig-Holstein,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((8.44242 55.05062, 8.46241 55.0..."


,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,NAME_ENGL,NAME_FREN,ISO3_CODE,SVRG_UN,CAPT,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry
0,DE933,3,DE,Harburg,Harburg,4,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((9.78358 53.5, 9.78321 53.49212..."
1,DE934,3,DE,Lüchow-Dannenberg,Lüchow-Dannenberg,4,3,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((10.97163 53.1981, 11.01166 53.18148,..."
2,DE935,3,DE,"Lüneburg, Landkreis","Lüneburg, Landkreis",4,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((10.55863 53.36796, 10.57365 53..."
3,DEA5A,3,DE,Siegen-Wittgenstein,Siegen-Wittgenstein,2,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((8.49703 51.13709, 8.50502 51.13271, ..."
4,DEA5B,3,DE,Soest,Soest,4,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((8.33538 51.72791, 8.34204 51.72319, ..."


##### Inspect Boundary Attributes

In [15]:
# ============================================================
# 3. Inspect boundary attributes
# ============================================================

print(f"NUTS-2 regions: {len(nuts2)}")
print(f"NUTS-3 districts: {len(nuts3)}")

print("\nCoordinate Reference System (CRS)")
print("NUTS-2:", nuts2.crs)
print("NUTS-3:", nuts3.crs)

print("\nKey attributes")
print("NUTS-2:", ["NUTS_ID", "LEVL_CODE", "CNTR_CODE", "NUTS_NAME", "geometry"])
print("NUTS-3:", ["NUTS_ID", "LEVL_CODE", "CNTR_CODE", "NUTS_NAME", "geometry"])

print("\nCountry codes")
print("NUTS-2:", nuts2["CNTR_CODE"].unique())
print("NUTS-3:", nuts3["CNTR_CODE"].unique())

NUTS-2 regions: 38
NUTS-3 districts: 401

Coordinate Reference System (CRS)
NUTS-2: EPSG:4326
NUTS-3: EPSG:4326

Key attributes
NUTS-2: ['NUTS_ID', 'LEVL_CODE', 'CNTR_CODE', 'NUTS_NAME', 'geometry']
NUTS-3: ['NUTS_ID', 'LEVL_CODE', 'CNTR_CODE', 'NUTS_NAME', 'geometry']

Country codes
NUTS-2: ['DE']
NUTS-3: ['DE']


## Build the NUTS-2 → NUTS-3 Area-Weight Lookup Table

This step constructs the spatial lookup table that links Germany's NUTS-2 regions with their corresponding NUTS-3 (Kreise) districts through area-weighted spatial intersections. The lookup table provides the spatial relationship required to transfer climate indicators from the regional (NUTS-2) level to the district (NUTS-3) level in a transparent and reproducible manner.

The resulting lookup table serves as the foundation for harmonizing all climate indicators that are currently available only at the NUTS-2 regional level. By calculating the proportion of each NUTS-3 district that overlaps with a NUTS-2 region, we derive area-weighting factors that are subsequently used to disaggregate regional climate indicators to the district level.

### Methodological Justification

Area-weighted interpolation was selected because the climate indicators used in this project are available as aggregated regional statistics at the NUTS-2 level, while the target modelling framework operates at the finer NUTS-3 (Kreise) level. In the absence of higher-resolution climate observations, area-weighted interpolation provides a transparent, reproducible, and computationally efficient approach for spatial harmonization.

### Methodological Assumptions

The harmonization assumes that each climate indicator represents a regional average that is spatially homogeneous within each NUTS-2 region. Consequently, the climate value assigned to a NUTS-3 district is estimated using the proportional area of overlap between the district and its parent NUTS-2 region.

The resulting area-weight lookup table is generated once and subsequently reused for all climate indicators (heatwave days, flood proxy (Rx5day), and drought magnitude), ensuring methodological consistency, reproducibility, and computational efficiency across all subsequent climate harmonization workflows.

In [27]:
# ============================================================
# 4. Build NUTS-2 → NUTS-3 area-weight lookup table
# ============================================================

# Use a projected CRS for accurate area calculations.
# EPSG:3035 is commonly used for Europe-wide equal-area analysis.
nuts2_proj = nuts2.to_crs(epsg=3035)
nuts3_proj = nuts3.to_crs(epsg=3035)

print("Projected CRS")
print("NUTS-2:", nuts2_proj.crs)
print("NUTS-3:", nuts3_proj.crs)

# Keep only required columns before overlay
nuts2_small = nuts2_proj[["NUTS_ID", "NUTS_NAME", "geometry"]].rename(
    columns={
        "NUTS_ID": "nuts2_id",
        "NUTS_NAME": "nuts2_name"
    }
)

nuts3_small = nuts3_proj[["NUTS_ID", "NUTS_NAME", "geometry"]].rename(
    columns={
        "NUTS_ID": "nuts3_id",
        "NUTS_NAME": "nuts3_name"
    }
)

# Calculate original NUTS-3 district area
nuts3_area = nuts3_small.copy()
nuts3_area["nuts3_area_m2"] = nuts3_area.geometry.area
nuts3_area = nuts3_area[["nuts3_id", "nuts3_area_m2"]]

# Compute spatial intersections between NUTS-3 and NUTS-2
intersections = gpd.overlay(
    nuts3_small,
    nuts2_small,
    how="intersection",
    keep_geom_type=False
)

# Calculate overlap area
intersections["overlap_area_m2"] = intersections.geometry.area

# Attach original NUTS-3 area and calculate area weight
area_weights = intersections.merge(nuts3_area, on="nuts3_id", how="left")
area_weights["area_weight"] = (
    area_weights["overlap_area_m2"] / area_weights["nuts3_area_m2"]
)

# Keep relevant columns only
area_weights = area_weights[
    [
        "nuts2_id",
        "nuts2_name",
        "nuts3_id",
        "nuts3_name",
        "overlap_area_m2",
        "nuts3_area_m2",
        "area_weight"
    ]
].copy()

# Sort for readability
area_weights = area_weights.sort_values(["nuts2_id", "nuts3_id"]).reset_index(drop=True)

print("Area-weight lookup table shape:", area_weights.shape)
display(area_weights.head())

Projected CRS
NUTS-2: EPSG:3035
NUTS-3: EPSG:3035
Area-weight lookup table shape: (865, 7)


,nuts2_id,nuts2_name,nuts3_id,nuts3_name,overlap_area_m2,nuts3_area_m2,area_weight
0,DE11,Stuttgart,DE111,"Stuttgart, Stadtkreis",2.101117e+08,2.101117e+08,1.0
1,DE11,Stuttgart,DE112,Böblingen,6.167061e+08,6.167061e+08,1.0
2,DE11,Stuttgart,DE113,Esslingen,6.396529e+08,6.396529e+08,1.0
3,DE11,Stuttgart,DE114,Göppingen,6.421750e+08,6.421750e+08,1.0
4,DE11,Stuttgart,DE115,Ludwigsburg,6.859905e+08,6.859905e+08,1.0


##### Initial Validation of the Area-Weight Lookup Table

In [28]:
# ============================================================
# 5a. Validate the area-weight lookup table
# ============================================================

district_counts = (
    area_weights
    .groupby("nuts3_id")
    .size()
    .value_counts()
    .sort_index()
)

print(district_counts)

1    116
2    148
3    100
4     32
5      5
Name: count, dtype: int64


##### Inspect districts with multiple NUTS-2 intersections

In [29]:
# ============================================================
# 5b. Inspect districts with multiple NUTS-2 intersections
# ============================================================

multiple = (
    area_weights
    .groupby("nuts3_id")
    .filter(lambda x: len(x) > 1)
)

display(
    multiple.sort_values(["nuts3_id", "area_weight"])
)

,nuts2_id,nuts2_name,nuts3_id,nuts3_name,overlap_area_m2,nuts3_area_m2,area_weight
29,DE12,Karlsruhe,DE112,Böblingen,0.000000e+00,6.167061e+08,0.0
73,DE14,Tübingen,DE112,Böblingen,0.000000e+00,6.167061e+08,0.0
1,DE11,Stuttgart,DE112,Böblingen,6.167061e+08,6.167061e+08,1.0
74,DE14,Tübingen,DE113,Esslingen,0.000000e+00,6.396529e+08,0.0
2,DE11,Stuttgart,DE113,Esslingen,6.396529e+08,6.396529e+08,1.0
...,...,...,...,...,...,...,...
862,DEG0,Thüringen,DEG0M,Altenburger Land,5.698137e+08,5.698137e+08,1.0
400,DE73,Kassel,DEG0N,"Eisenach, Kreisfreie Stadt",0.000000e+00,1.046425e+08,0.0
863,DEG0,Thüringen,DEG0N,"Eisenach, Kreisfreie Stadt",1.046425e+08,1.046425e+08,1.0
401,DE73,Kassel,DEG0P,Wartburgkreis,0.000000e+00,1.305336e+09,0.0


**Interpretation**

Inspection of the multiple matches shows that the additional NUTS-2 assignments all have an area weight of 0.0, indicating that they arise solely from shared administrative boundaries rather than genuine spatial overlap. These records can therefore be safely removed without affecting the integrity of the lookup table.

##### Remove Zero-Area Boundary Intersections

The initial spatial overlay identifies all geometric intersections between NUTS-2 and NUTS-3 polygons. Some of these correspond only to shared administrative boundaries and therefore have zero overlap area. These boundary artefacts are removed before constructing the final lookup table to ensure that each NUTS-3 district is associated only with its true parent NUTS-2 region.

In [30]:
area_weights = area_weights.query("area_weight > 0").copy()

##### Final Validation of the Area-Weight Lookup Table

In [31]:
# ============================================================
# 5c. Validate the area-weight lookup table
# ============================================================

district_counts = (
    area_weights
    .groupby("nuts3_id")
    .size()
    .value_counts()
    .sort_index()
)

print(district_counts)

1    401
Name: count, dtype: int64


**Validation Outcome**

After removing zero-area boundary artefacts, each of Germany's 401 NUTS-3 districts is associated with exactly one NUTS-2 region. This confirms that the validated area-weight lookup table is suitable for subsequent climate data harmonization.

#### Save the Validated Area-Weight Lookup Table

The validated NUTS-2 → NUTS-3 area-weight lookup table is saved as a reusable dataset. This lookup table will be used by all subsequent climate harmonization workflows to transfer climate indicators from the NUTS-2 regional level to Germany's NUTS-3 districts using consistent area-weight interpolation.

In [33]:
# ================================================================
# 7. Save the validated area-weight lookup table
# ================================================================

OUTPUT_DIR = BASE_DIR / "data" / "processed" / "climate" / "spatial_lookup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

area_weights.to_csv(
    OUTPUT_DIR / "nuts2_to_nuts3_area_weights.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Lookup table saved to: {OUTPUT_DIR}")
print(f"Records: {len(area_weights):,}")

Lookup table saved to: d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\processed\climate\spatial_lookup
Records: 401


## Summary

In this notebook, a reusable NUTS-2 → NUTS-3 spatial area-weight lookup table was constructed and validated. The lookup table provides the spatial foundation required to harmonize climate indicators available at the NUTS-2 regional level to Germany's NUTS-3 districts in subsequent processing workflows.

The validated lookup table will be integrated with the climate datasets, enabling all climate indicators available at the NUTS-2 regional level to be harmonized to the NUTS-3 (district) level using consistent area-weight interpolation. The resulting harmonized climate datasets will provide the spatial foundation for integrating climate hazards with socioeconomic indicators and ultimately constructing the integrated Climate–ESG Risk Intelligence modelling dataset.

## Next Step

The validated lookup table will now be integrated into the heatwave, flood (Rx5day), and drought preprocessing workflows. Each climate indicator available at the NUTS-2 level will be transferred to the NUTS-3 district level using the validated area-weight interpolation approach, forming the spatial foundation of the integrated Climate–ESG Risk Intelligence dataset.

# Part II — Harmonize Climate Datasets

#### 2.1 Harmonize Heatwave Dataset

(code...)

#### 2.2 Harmonize Flood (Rx5day) Dataset

(code...)

#### 2.3 Harmonize Drought Dataset

(code...)

# Part III — Export Harmonized Climate Datasets


#### 3.1 Export Harmonized Heatwave Dataset

#### 3.2 Export Harmonized Flood (Rx5day) Dataset

#### 3.3 Export Harmonized Drought Dataset

#### 3.4 Export Combined Climate Dataset (Optional)